## Įvadas 

Ši pamoka apims: 
- Kas yra funkcijos kvietimas ir jo naudojimo atvejai 
- Kaip sukurti funkcijos kvietimą naudojant Azure OpenAI 
- Kaip integruoti funkcijos kvietimą į programą 

## Mokymosi tikslai 

Baigę šią pamoką, žinosite kaip ir suprasite: 

- Funkcijos kvietimo naudojimo tikslą 
- Kaip nustatyti funkcijos kvietimą naudojant Azure Open AI paslaugą 
- Kaip sukurti veiksmingus funkcijos kvietimus pagal jūsų programos naudojimo atvejį 


## Funkcijų kvietimų supratimas

Šiam užsiėmimui norime sukurti funkciją mūsų švietimo startuoliui, leidžiančią vartotojams naudoti pokalbių robotą techninių kursų paieškai. Rekomenduosime kursus, atitinkančius jų įgūdžių lygį, dabartinę rolę ir dominančią technologiją.

Tam įgyvendinti naudosime šių kombinaciją:
 - `Azure Open AI` kuriant pokalbių patirtį vartotojui
 - `Microsoft Learn Catalog API`, padedančią vartotojams rasti kursus pagal jų užklausą
 - `Funkcijų kvietimą` norint vartotojo užklausą perduoti funkcijai, kuri atliktų API užklausą.

Norėdami pradėti, pažvelkime, kodėl iš viso norėtume naudoti funkcijų kvietimą:


### Kodėl reikalingas funkcijų kvietimas

Jei jau baigėte bet kurią kitą šio kurso pamoką, greičiausiai suprantate Didžiųjų kalbos modelių (LLM) naudojimo galią. Tikimės, kad taip pat galite matyti kai kurias jų ribotumus.

Funkcijų kvietimas yra Azure Open AI paslaugos funkcija, skirta įveikti šias ribotumus:
1) Nuosekli atsakymų forma
2) Galimybė naudoti duomenis iš kitų programos šaltinių pokalbio kontekste

Prieš funkcijų kvietimą LLM atsakymai buvo nestruktūruoti ir nenuspėjami. Kūrėjai turėjo rašyti sudėtingą tikrinimo kodą, kad įsitikintų, jog gali apdoroti kiekvieną atsakymo variaciją.

Vartotojai negalėjo gauti atsakymų į klausimus, pavyzdžiui, „Koks dabar oras Stokholme?“. Taip yra todėl, kad modeliai buvo apriboti mokymosi metu turimais duomenimis.

Pažiūrėkime į žemiau pateiktą pavyzdį, kuris iliustruoja šią problemą:

Tarkime, norime sukurti studentų duomenų bazę, kad galėtume jiems pasiūlyti tinkamus kursus. Žemiau pateikiamos dvi studentų aprašų versijos, kurios yra labai panašios savo turimuose duomenyse.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Mes norime tai siųsti dideliam kalbos modeliui (LLM), kad jis išanalizuotų duomenis. Vėliau tai gali būti naudojama mūsų programoje, norint siųsti tai į API arba saugoti duomenų bazėje. 

Sukurkime du identiškus užklausimus, kuriuose nurodome LLM, kokia informacija mus domina: 


Norime tai išsiųsti LLM, kad jis išanalizuotų mūsų produktui svarbias dalis. Taigi galime sukurti du identiškus užklausimus, kad nurodytume LLM: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Sukūrę šiuos du užklausimus, mes juos siunčiame LLM naudodami `client.responses.create`. Užklausimą saugome kintamajame `input` ir priskiriame vaidmenį `user`. Tai imituoja vartotojo žinutės rašymą pokalbių botui. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Dabar galime nusiųsti abu užklausimus LLM ir išnagrinėti gautą atsakymą. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Nors užklausos yra tos pačios ir aprašymai panašūs, galime gauti skirtingus `Grades` savybės formatus.

Jei aukščiau esantį langelį paleisite kelis kartus, formatas gali būti `3.7` arba `3.7 GPA`.

Taip yra todėl, kad LLM priima nestruktūruotus duomenis rašyto prašymo formoje ir taip pat pateikia nestruktūruotus duomenis. Mums reikia turėti struktūruotą formatą, kad žinotume, ko tikėtis saugant ar naudojant šiuos duomenis.

Naudodamiesi funkciniu kvietimu galime užtikrinti, kad gausime struktūruotus duomenis atgal. Naudojant funkcijų kvietimą, LLM iš tikrųjų nekviečia ar nevykdo jokių funkcijų. Vietoje to mes sukuriame struktūrą, kurios turi laikytis LLM atsakymuose. Tada naudojame tuos struktūruotus atsakymus, kad žinotume, kurią funkciją paleisti mūsų programose.
 


![Funkcijos kvietimo srauto diagrama](../../../../translated_images/lt/Function-Flow.083875364af4f4bb.webp)


Tada galime paimti tai, kas grąžinta iš funkcijos, ir išsiųsti tai atgal į LLM. LLM tuomet atsakys natūralia kalba, kad atsakytų į vartotojo užklausą. 


### Naudojimo atvejai funkcijų kvietimams

**Išorinių įrankių kvietimas**
Pokalbių robotai puikiai atsako į vartotojų klausimus. Naudojant funkcijų kvietimus, pokalbių robotai gali panaudoti vartotojų žinutes tam tikroms užduotims atlikti. Pavyzdžiui, studentas gali paprašyti pokalbių roboto „Išsiųsk el. laišką mano dėstytojui, kad man reikia daugiau pagalbos su šia tema“. Tai gali sukelti funkcijos kvietimą `send_email(to: string, body: string)`


**API ar duomenų bazių užklausų kūrimas**
Vartotojai gali rasti informaciją naudodami natūralią kalbą, kuri virsta suformatuota užklausa arba API užklausa. Pavyzdys galėtų būti mokytojas, kuris užduoda klausimą „Kas yra studentai, kurie atliko paskutinį užduotį“, ir tai gali iškviesti funkciją `get_completed(student_name: string, assignment: int, current_status: string)`


**Struktūrizuotų duomenų kūrimas**
Vartotojai gali paimti teksto bloką arba CSV ir naudoti LLM ištraukti svarbią informaciją. Pavyzdžiui, studentas gali paversti Vikipedijos straipsnį apie taikos susitarimus AI kortelėmis. Tai galima padaryti naudojant funkciją `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Pirmojo funkcijos iškvietimo sukūrimas 

Funkcijos iškvietimo sukūrimo procesas apima 3 pagrindinius žingsnius: 
1. Iškvietimas Chat Completions API su jūsų funkcijų sąrašu ir naudotojo žinute 
2. Perskaityti modelio atsakymą, kad būtų galima atlikti veiksmą, pvz., vykdyti funkciją arba API iškvietimą 
3. Atlikti dar vieną iškvietimą Chat Completions API su funkcijos atsakymu, kad būtų panaudota ši informacija atsakymui kurti naudotojui. 


![Funkcijos kvietimo srautas](../../../../translated_images/lt/LLM-Flow.3285ed8caf4796d7.webp)


### Funkcijos iškvietimo elementai

#### Vartotojo įvestis

Pirmas žingsnis yra sukurti vartotojo pranešimą. Jis gali būti dinamiškai priskiriamas paimant teksto įvesties reikšmę arba galite priskirti reikšmę čia. Jei tai jūsų pirmas kartas dirbant su Chat Completions API, turime apibrėžti pranešimo `role` ir `content`.

`role` gali būti `system` (taisyklėms kurti), `assistant` (modelis) arba `user` (galutinis vartotojas). Funkcijos iškvietimui mes tai priskirsime kaip `user` ir pateiksime pavyzdinį klausimą.


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Funkcijų kūrimas. 

Toliau apibrėšime funkciją ir tos funkcijos parametrus. Čia naudosime vieną funkciją, pavadintą `search_courses`, tačiau galite sukurti kelias funkcijas.

**Svarbu**: Funkcijos yra įtraukiamos į sistemos žinutę LLM ir bus skaičiuojamos į jūsų turimų žetonų kiekį.


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Apibrėžimai** 

`name` - Funkcijos, kurią norime iškviesti, pavadinimas. 

`description` - Aprašymas, kaip veikia funkcija. Čia svarbu būti konkrečiam ir aiškiam. 

`parameters` - Vertybių ir formato sąrašas, kurį norite, kad modelis sugeneruotų savo atsakyme. 


`type` - Duomenų tipas, kuriame bus saugomos savybės. 

`properties` - Konkretų verčių sąrašas, kurį modelis naudos savo atsakyme. 


`name` - Savybės pavadinimas, kurį modelis naudos suformatuotame atsakyme. 

`type` - Šios savybės duomenų tipas. 

`description` - Konkretčios savybės aprašymas. 


**Pasirenkama**

`required` - Privaloma savybė, kad funkcijos iškvietimas būtų užbaigtas. 


### Funkcijos iškvietimas 
Apibrėžus funkciją, dabar ją reikia įtraukti į kvietimą Chat Completion API. Tai atliekama pridedant `functions` prie užklausos. Šiuo atveju `functions=functions`. 

Taip pat yra galimybė nustatyti `function_call` kaip `auto`. Tai reiškia, kad leisime LLM nuspręsti, kuri funkcija turėtų būti iškviesta pagal vartotojo pranešimą, o ne priskirsime ją patys.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Dabar pažvelkime į atsakymą ir pamatykime, kaip jis yra suformatuotas: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Matote, kad funkcijos pavadinimas yra iškviečiamas, o iš vartotojo pranešimo LLM sugebėjo rasti duomenis, atitinkančius funkcijos argumentus. 


## 3. Funkcijų iškvietimų integravimas į programą. 


Po to, kai išbandėme suformatuotą atsakymą iš LLM, dabar galime tai integruoti į programą. 

### Srauto valdymas 

Kad tai integruotume į mūsų programą, atlikime šiuos veiksmus: 

Pirma, atlikime skambutį Open AI paslaugoms ir išsaugokime žinutę kintamajame `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Dabar apibrėšime funkciją, kuri iškvies Microsoft Learn API, kad gautų kursų sąrašą: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Kaip geriausia praktika, tada pažiūrėsime, ar modelis nori iškviesti funkciją. Po to mes sukursime vieną iš prieinamų funkcijų ir suderinsime ją su kviečiama funkcija. 
Tada paimsime funkcijos argumentus ir susiesime juos su LLM argumentais.

Galiausiai pridėsime funkcijos kvietimo žinutę ir reikšmes, kurias grąžino `search_courses` žinutė. Tai suteikia LLM visą informaciją, kurios jam reikia,
kad galėtų natūralia kalba atsakyti vartotojui. 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Dabar mes išsiųsime atnaujintą žinutę į LLM, kad galėtume gauti natūralios kalbos atsakymą, o ne API JSON formato atsakymą. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Kodo Iššūkis 

Puikus darbas! Norėdami tęsti mokymąsi apie Azure Open AI funkcijų kvietimą, galite sukurti: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - Daugiau funkcijos parametrų, kurie gali padėti besimokantiesiems rasti daugiau kursų. Galimus API parametrus rasite čia: 
 - Sukurkite dar vieną funkcijos kvietimą, kuris priima daugiau informacijos iš besimokančiojo, pvz., jų gimtąją kalbą 
 - Sukurkite klaidų tvarkymą, kai funkcijos kvietimas ir/ar API kvietimas negrąžina tinkamų kursų 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
